# La matemática de autograd: regla de la cadena y retropropagación

Este módulo recorre las funciones de PyTorch que usamos en el curso y, para cada una, hace tres cosas:

1. Escribe **la fórmula** que implementa.
2. **Justifica** de dónde sale.
3. La **reimplementa a mano** y comprueba con `torch.allclose` que da lo mismo que PyTorch.

Empezamos por la pieza de la que depende todo lo demás: `backward()`. Al terminar este notebook habrás calculado a mano, con matrices, los mismos gradientes que entrega autograd para una red con capa oculta.

**Contenido**

1. Derivada, gradiente y jacobiano
2. La regla de la cadena
3. El grafo computacional
4. Modo reverso: productos vector-jacobiano
5. Por qué `backward()` pide un escalar
6. Por qué los gradientes se acumulan
7. Las reglas locales de cada operación
8. Retropropagación a mano en una red
9. Verificación numérica con diferencias finitas
10. `detach()` y `no_grad()`: cortar el grafo
11. Resumen y ejercicios

Los cálculos de este módulo usan `float64` para que las comparaciones entre "a mano" y PyTorch no se ensucien con errores de redondeo.

In [1]:
import torch
from torch.autograd.functional import jacobian

torch.manual_seed(0)
torch.set_default_dtype(torch.float64)
print("Versión de PyTorch:", torch.__version__)

Versión de PyTorch: 2.11.0+cpu


## 1. Derivada, gradiente y jacobiano

Los tres conceptos son el mismo, para funciones con distinta cantidad de entradas y salidas.

**Derivada.** Para $f: \mathbb{R} \to \mathbb{R}$, mide cuánto cambia la salida por unidad de cambio en la entrada:

$$f'(x) = \lim_{\varepsilon \to 0} \frac{f(x + \varepsilon) - f(x)}{\varepsilon}$$

**Gradiente.** Para $f: \mathbb{R}^n \to \mathbb{R}$ (muchas entradas, una salida), es el vector de las derivadas parciales:

$$\nabla f(x) = \left( \frac{\partial f}{\partial x_1}, \dots, \frac{\partial f}{\partial x_n} \right)$$

Una función de pérdida es de este tipo: recibe todos los parámetros y devuelve un número.

**Jacobiano.** Para $f: \mathbb{R}^n \to \mathbb{R}^m$ (muchas entradas, muchas salidas), es la matriz de $m \times n$ con todas las derivadas parciales. La fila $i$ es el gradiente de la salida $i$:

$$J_{ij} = \frac{\partial f_i}{\partial x_j}$$

Una capa de una red es de este tipo. El jacobiano es la mejor aproximación lineal de la función cerca de un punto: $f(x + \delta) \approx f(x) + J \delta$.

`torch.autograd.functional.jacobian` lo calcula completo. Ejemplo con $f(x_1, x_2) = (x_1 x_2, \; x_1 + x_2, \; x_1^2)$:

In [2]:
def f(x):
    return torch.stack([x[0] * x[1], x[0] + x[1], x[0] ** 2])


x = torch.tensor([2.0, 3.0])

J = jacobian(f, x)
J_a_mano = torch.tensor([[x[1], x[0]],          # derivadas de x1*x2
                         [1.0, 1.0],            # derivadas de x1+x2
                         [2 * x[0], 0.0]])      # derivadas de x1^2

print("Jacobiano con PyTorch:\n", J)
print("¿Coincide con el calculado a mano?", torch.allclose(J, J_a_mano))

Jacobiano con PyTorch:
 tensor([[3., 2.],
        [1., 1.],
        [4., 0.]])
¿Coincide con el calculado a mano? True


## 2. La regla de la cadena

Una red neuronal es una composición de funciones: la salida de una capa es la entrada de la siguiente. La regla de la cadena dice cómo derivar una composición.

**Una variable.** Si $y = f(u)$ y $u = g(x)$:

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

Las tasas de cambio se multiplican: si $u$ cambia 3 veces más rápido que $x$, e $y$ cambia 2 veces más rápido que $u$, entonces $y$ cambia 6 veces más rápido que $x$.

**Varias variables.** Si $y = f(u)$ y $u = g(x)$ con vectores, los jacobianos se multiplican como matrices:

$$J_{f \circ g}(x) = J_f(g(x)) \; J_g(x)$$

Elemento a elemento, esto dice que hay que **sumar sobre todos los caminos intermedios**:

$$\frac{\partial y_i}{\partial x_j} = \sum_k \frac{\partial y_i}{\partial u_k} \frac{\partial u_k}{\partial x_j}$$

Esa suma es la que explica, en la sección 6, por qué los gradientes se acumulan.

In [3]:
def g(x):
    return torch.stack([x[0] * x[1], x[0] + x[1], x[0] ** 2])       # R^2 -> R^3


def f(u):
    return torch.stack([u[0] * u[2], torch.sin(u[1])])              # R^3 -> R^2


x = torch.tensor([2.0, 3.0])

J_compuesta = jacobian(lambda v: f(g(v)), x)        # jacobiano de la composición, directo
J_producto = jacobian(f, g(x)) @ jacobian(g, x)     # producto de los jacobianos

print("Formas:", tuple(jacobian(f, g(x)).shape), "@", tuple(jacobian(g, x).shape), "->", tuple(J_producto.shape))
print("¿J de la composición = producto de jacobianos?", torch.allclose(J_compuesta, J_producto))

Formas: (2, 3) @ (3, 2) -> (2, 2)
¿J de la composición = producto de jacobianos? True


## 3. El grafo computacional

Cuando un tensor tiene `requires_grad=True`, PyTorch anota cada operación que se hace con él. El resultado es un **grafo**: los nodos son tensores y cada arista recuerda qué operación produjo qué.

Para la pérdida de una neurona en un solo ejemplo, $L = (w x + b - y)^2$, el cálculo se descompone en pasos elementales:

```
w ──┐
    ├─(×)── p ──┐
x ──┘           ├─(+)── s ──┐
b ──────────────┘           ├─(−)── e ──(²)── L
y ──────────────────────────┘
```

Cada tensor resultante guarda en `grad_fn` la operación que lo creó, y desde ahí se puede caminar hacia atrás:

In [4]:
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(2.0)
y = torch.tensor(5.0)

p = w * x
s = p + b
e = s - y
L = e ** 2

nodo = L.grad_fn
while nodo is not None:
    print(type(nodo).__name__)
    entradas = nodo.next_functions
    nodo = entradas[0][0] if entradas else None     # seguir la primera entrada de cada operación

PowBackward0
SubBackward0
AddBackward0
MulBackward0
AccumulateGrad


Leído de arriba abajo es el cálculo al revés: potencia, resta, suma, multiplicación y, al final, `AccumulateGrad`, que es el nodo donde se deposita el gradiente de una hoja (`w`).

Para derivar $L$ respecto de $w$ se recorre el grafo desde $L$ hacia atrás, multiplicando la derivada local de cada operación:

$$\frac{\partial L}{\partial w} = \underbrace{\frac{\partial L}{\partial e}}_{2e} \cdot \underbrace{\frac{\partial e}{\partial s}}_{1} \cdot \underbrace{\frac{\partial s}{\partial p}}_{1} \cdot \underbrace{\frac{\partial p}{\partial w}}_{x} = 2 e x$$

Cada factor es la derivada de una operación elemental, que PyTorch trae programada. Nadie tuvo que derivar la expresión completa.

In [5]:
L.backward()

print("e =", e.item())
print(f"dL/dw   autograd: {w.grad.item():6.2f}   a mano (2·e·x): {(2 * e * x).item():6.2f}")
print(f"dL/db   autograd: {b.grad.item():6.2f}   a mano (2·e)  : {(2 * e).item():6.2f}")

e = 2.0
dL/dw   autograd:   8.00   a mano (2·e·x):   8.00
dL/db   autograd:   4.00   a mano (2·e)  :   4.00


## 4. Modo reverso: productos vector-jacobiano

Supón una cadena de tres funciones que termina en un escalar:

$$x \;\xrightarrow{\;f_1\;}\; u_1 \;\xrightarrow{\;f_2\;}\; u_2 \;\xrightarrow{\;f_3\;}\; L$$

Por la regla de la cadena, el gradiente es un producto de jacobianos (lo escribimos como vector fila):

$$\nabla L^\top = J_3 \, J_2 \, J_1$$

Ese producto se puede calcular en dos órdenes, y el costo es muy distinto.

- **De derecha a izquierda** (modo directo): se parte de $J_1$, que tiene tantas columnas como entradas. Para obtener el gradiente completo hay que arrastrar matrices enteras o, lo que es equivalente, repetir el cálculo una vez por cada entrada.
- **De izquierda a derecha** (modo reverso): como $L$ es un escalar, $J_3$ es una sola fila. El producto $J_3 J_2$ es otra fila, y $(J_3 J_2) J_1$ otra. En ningún momento se construye una matriz: solo se va transformando **un vector**.

`backward()` usa el modo reverso. En cada operación del grafo recibe el vector $v = \partial L / \partial(\text{salida})$ y calcula

$$v^\top J = \frac{\partial L}{\partial(\text{entrada})}$$

A esto se le llama **producto vector-jacobiano** (VJP). PyTorch nunca arma el jacobiano $J$ de una capa: cada operación solo sabe hacer este producto, de forma eficiente.

La consecuencia es la razón de ser del aprendizaje profundo: **una sola pasada hacia atrás entrega la derivada de la pérdida respecto de todos los parámetros**, con un costo comparable al de una pasada hacia adelante, sean diez parámetros o mil millones. El modo directo necesitaría una pasada por parámetro.

Se puede pedir un VJP directamente con `torch.autograd.grad` y el argumento `grad_outputs`. Lo comparamos con multiplicar por el jacobiano explícito:

In [6]:
def capa(x):
    return torch.tanh(x) * x.sum()          # una función cualquiera de R^4 -> R^4


x = torch.randn(4, requires_grad=True)
v = torch.randn(4)                          # hace el papel de dL/d(salida)

vjp = torch.autograd.grad(capa(x), x, grad_outputs=v)[0]        # sin construir J
J = jacobian(capa, x.detach())                                  # J completo, 4x4

print("VJP con autograd:", vjp)
print("v @ J explícito :", v @ J)
print("¿Coinciden?", torch.allclose(vjp, v @ J))

VJP con autograd: tensor([-0.4866, -0.0865, -0.5599, -0.7763])
v @ J explícito : tensor([-0.4866, -0.0865, -0.5599, -0.7763])
¿Coinciden? True


## 5. Por qué `backward()` pide un escalar

Si llamas a `backward()` sobre un tensor con varios elementos, PyTorch responde `grad can be implicitly created only for scalar outputs`. No es una limitación arbitraria.

El modo reverso necesita un vector inicial $v$ con el que empezar la cadena de productos. Cuando la salida es un escalar $L$, hay una elección natural: $v = \partial L / \partial L = 1$. Cuando la salida es un vector, no hay un $v$ obvio: hay que decir **qué combinación** de las salidas se quiere derivar.

Por eso `y.backward(v)` acepta ese vector como argumento. El resultado es el gradiente de la función escalar $v^\top y$. Usar `v` lleno de unos equivale a derivar la suma:

In [7]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = x ** 2
y.backward(torch.ones(3))               # v = (1, 1, 1)
grad_con_v = x.grad.clone()

x.grad.zero_()
(x ** 2).sum().backward()               # derivar la suma, que sí es un escalar

print("backward con v de unos:", grad_con_v)
print("backward de la suma   :", x.grad)

backward con v de unos: tensor([2., 4., 6.])
backward de la suma   : tensor([2., 4., 6.])


En la práctica la pérdida que se deriva es un escalar (por eso las funciones de pérdida promedian o suman sobre el lote), y `backward()` se llama sin argumentos.

## 6. Por qué los gradientes se acumulan

La regla de la cadena en varias variables suma sobre todos los caminos por los que una variable influye en el resultado. Si un mismo tensor se usa en dos partes del cálculo, su gradiente es la **suma** de las dos contribuciones.

Ejemplo: $y = x^2 + x^3$. La variable $x$ entra por dos caminos, y

$$\frac{dy}{dx} = \underbrace{2x}_{\text{camino } x^2} + \underbrace{3x^2}_{\text{camino } x^3}$$

Para implementar esto, cada vez que el recorrido hacia atrás llega a un tensor por algún camino, PyTorch **suma** la contribución a su `.grad` en lugar de reemplazarla. Ese es el nodo `AccumulateGrad` de la sección 3.

In [8]:
x = torch.tensor(2.0, requires_grad=True)

a = x ** 2
b = x ** 3
y = a + b
y.backward()

print("camino x²:", (2 * x).item(), "   camino x³:", (3 * x ** 2).item())
print("x.grad   :", x.grad.item(), "  (la suma de los dos)")

camino x²: 4.0    camino x³: 12.0
x.grad   : 16.0   (la suma de los dos)


El mismo mecanismo actúa entre llamadas distintas a `backward()`: PyTorch no sabe si la nueva llamada es "otro camino" del mismo cálculo o un cálculo nuevo, así que suma igual. Por eso en un ciclo de entrenamiento hay que llamar a `zero_grad()` en cada paso. Lo que parece una incomodidad es la consecuencia directa de implementar bien la regla de la cadena.

Este comportamiento también tiene usos deliberados: llamar a `backward()` sobre varios lotes pequeños antes de un solo `step()` acumula sus gradientes, lo que simula un lote más grande cuando no cabe en memoria (dividiendo cada pérdida por la cantidad de lotes acumulados, para obtener el promedio y no la suma).

## 7. Las reglas locales de cada operación

Cada operación de PyTorch implementa dos cosas: cómo calcular su salida, y cómo convertir el gradiente respecto de su salida en el gradiente respecto de sus entradas (su VJP). Llamamos $G$ al gradiente que llega desde adelante, $G = \partial L / \partial(\text{salida})$, que tiene la misma forma que la salida.

| Operación | Salida | Gradiente hacia las entradas |
|---|---|---|
| Suma | $C = A + B$ | $\partial L / \partial A = G$, $\quad \partial L / \partial B = G$ |
| Producto elemento a elemento | $C = A \odot B$ | $\partial L / \partial A = G \odot B$, $\quad \partial L / \partial B = G \odot A$ |
| Producto matricial | $C = A B$ | $\partial L / \partial A = G B^\top$, $\quad \partial L / \partial B = A^\top G$ |
| Función elemento a elemento | $C = \phi(A)$ | $\partial L / \partial A = G \odot \phi'(A)$ |
| Suma de todos los elementos | $c = \sum_{ij} A_{ij}$ | $\partial L / \partial A = G \cdot \mathbf{1}$ (se reparte a todos) |
| Promedio | $c = \frac{1}{n} \sum_{ij} A_{ij}$ | $\partial L / \partial A = \frac{G}{n} \cdot \mathbf{1}$ |

Dos de estas reglas merecen justificación.

**Producto matricial.** Con $C = AB$, cada elemento es $C_{ij} = \sum_k A_{ik} B_{kj}$. El elemento $A_{ik}$ aparece en todos los $C_{ij}$ de la fila $i$, con coeficiente $B_{kj}$. Sumando sobre esos caminos:

$$\frac{\partial L}{\partial A_{ik}} = \sum_j \frac{\partial L}{\partial C_{ij}} \frac{\partial C_{ij}}{\partial A_{ik}} = \sum_j G_{ij} B_{kj} = (G B^\top)_{ik}$$

El mismo razonamiento para $B$ da $A^\top G$. Una forma de recordarlo: las formas tienen que calzar. $G$ tiene la forma de $C$, y la única manera de combinar $G$ con $B$ para obtener algo con la forma de $A$ es $G B^\top$.

**Broadcasting.** Cuando un tensor se "estira" para sumarse con otro más grande, en realidad se está usando varias veces. Por la sección 6, su gradiente es la suma sobre todas las copias. Por eso el gradiente de un sesgo $b$ que se suma a cada fila de un lote es la **suma sobre las filas** de $G$.

Comprobamos la regla del producto matricial:

In [9]:
A = torch.randn(3, 4, requires_grad=True)
B = torch.randn(4, 2, requires_grad=True)
pesos = torch.randn(3, 2)

L = ((A @ B) * pesos).sum()         # una pérdida escalar cualquiera; aquí G = pesos
L.backward()

G = pesos
print("dL/dA = G Bᵀ ?", torch.allclose(A.grad, G @ B.T.detach()))
print("dL/dB = Aᵀ G ?", torch.allclose(B.grad, A.T.detach() @ G))

dL/dA = G Bᵀ ? True
dL/dB = Aᵀ G ? True


## 8. Retropropagación a mano en una red

Con esas reglas alcanza para derivar una red completa. Tomamos una red con una capa oculta y ReLU, y la pérdida MSE, sobre un lote de $N$ ejemplos.

**Hacia adelante.** PyTorch guarda los datos con un ejemplo por fila, y en `nn.Linear` la matriz de pesos tiene forma `[salidas, entradas]`. Con esa convención:

$$Z_1 = X W_1^\top + b_1 \qquad H = \text{ReLU}(Z_1) \qquad \hat{Y} = H W_2^\top + b_2 \qquad L = \frac{1}{n} \sum (\hat{Y} - Y)^2$$

donde $n$ es la cantidad total de elementos de $\hat{Y}$.

**Hacia atrás.** Se aplica la tabla de la sección 7, de la última operación a la primera. Para abreviar, $\delta_A$ denota $\partial L / \partial A$.

| Paso | Fórmula | Regla usada |
|---|---|---|
| 1 | $\delta_{\hat{Y}} = \frac{2}{n} (\hat{Y} - Y)$ | promedio y cuadrado |
| 2 | $\delta_{W_2} = \delta_{\hat{Y}}^\top \, H$ | producto matricial |
| 3 | $\delta_{b_2} = $ suma de $\delta_{\hat{Y}}$ sobre las filas | broadcasting |
| 4 | $\delta_{H} = \delta_{\hat{Y}} \, W_2$ | producto matricial |
| 5 | $\delta_{Z_1} = \delta_{H} \odot \mathbb{1}[Z_1 > 0]$ | función elemento a elemento (ReLU) |
| 6 | $\delta_{W_1} = \delta_{Z_1}^\top \, X$ | producto matricial |
| 7 | $\delta_{b_1} = $ suma de $\delta_{Z_1}$ sobre las filas | broadcasting |

Los pasos 2 y 6 salen de la regla del producto matricial aplicada a $X W^\top$: el gradiente respecto de $W^\top$ es $X^\top \delta$, y al trasponer queda $\delta^\top X$.

El paso 5 es la derivada de la ReLU: vale 1 donde la entrada fue positiva y 0 donde no. El gradiente pasa intacto por las neuronas activas y se bloquea en las apagadas.

Fíjate que cada paso reutiliza el resultado del anterior ($\delta_{\hat{Y}}$ se usa en 2, 3 y 4; $\delta_{Z_1}$ en 6 y 7). Esa reutilización es lo que hace eficiente a la retropropagación.

In [10]:
N, entradas, ocultas, salidas = 6, 3, 4, 2

X = torch.randn(N, entradas)
Y = torch.randn(N, salidas)

W1 = torch.randn(ocultas, entradas, requires_grad=True)
b1 = torch.randn(ocultas, requires_grad=True)
W2 = torch.randn(salidas, ocultas, requires_grad=True)
b2 = torch.randn(salidas, requires_grad=True)

# Hacia adelante
Z1 = X @ W1.T + b1
H = torch.relu(Z1)
Y_pred = H @ W2.T + b2
L = ((Y_pred - Y) ** 2).mean()

# Hacia atrás con autograd
L.backward()

# Hacia atrás a mano
with torch.no_grad():
    d_Y_pred = 2 * (Y_pred - Y) / Y_pred.numel()        # paso 1
    d_W2 = d_Y_pred.T @ H                               # paso 2
    d_b2 = d_Y_pred.sum(dim=0)                          # paso 3
    d_H = d_Y_pred @ W2                                 # paso 4
    d_Z1 = d_H * (Z1 > 0)                               # paso 5
    d_W1 = d_Z1.T @ X                                   # paso 6
    d_b1 = d_Z1.sum(dim=0)                              # paso 7

for nombre, a_mano, parametro in [("W1", d_W1, W1), ("b1", d_b1, b1), ("W2", d_W2, W2), ("b2", d_b2, b2)]:
    print(f"{nombre}: forma {tuple(a_mano.shape)}   ¿coincide con autograd? {torch.allclose(a_mano, parametro.grad)}")

W1: forma (4, 3)   ¿coincide con autograd? True
b1: forma (4,)   ¿coincide con autograd? True
W2: forma (2, 4)   ¿coincide con autograd? True
b2: forma (2,)   ¿coincide con autograd? True


Los cuatro gradientes coinciden. Lo que hace `backward()` en una red es exactamente esto: aplicar reglas locales, de atrás hacia adelante, reutilizando resultados intermedios.

Nota que cada gradiente tiene la **misma forma** que su parámetro. Tiene que ser así, porque la actualización $\theta \leftarrow \theta - \eta \, \partial L / \partial \theta$ resta uno del otro.

## 9. Verificación numérica con diferencias finitas

¿Cómo saber que una derivada está bien sin confiar en ninguna fórmula? Volviendo a la definición: mover un parámetro un poquito y medir cuánto cambia la función.

La aproximación **centrada** es más precisa que la de un solo lado:

$$\frac{\partial f}{\partial \theta_i} \approx \frac{f(\theta + \varepsilon e_i) - f(\theta - \varepsilon e_i)}{2 \varepsilon}$$

donde $e_i$ es el vector con un 1 en la posición $i$. Su error es del orden de $\varepsilon^2$, porque al restar las dos expansiones de Taylor los términos pares se cancelan.

Esto requiere dos evaluaciones de la función **por cada parámetro**, por lo que no sirve para entrenar. Pero es la forma estándar de comprobar que un gradiente está bien calculado.

In [11]:
def perdida_de(W1_):
    """La pérdida de la red como función solo de W1."""
    H_ = torch.relu(X @ W1_.T + b1.detach())
    return (((H_ @ W2.detach().T + b2.detach()) - Y) ** 2).mean()


def gradiente_numerico(funcion, theta, eps=1e-6):
    grad = torch.zeros_like(theta)
    for i in range(theta.numel()):
        desplazamiento = torch.zeros_like(theta)
        desplazamiento.view(-1)[i] = eps
        grad.view(-1)[i] = (funcion(theta + desplazamiento) - funcion(theta - desplazamiento)) / (2 * eps)
    return grad


numerico = gradiente_numerico(perdida_de, W1.detach())

print("Diferencia máxima entre diferencias finitas y autograd:", (numerico - W1.grad).abs().max().item())

Diferencia máxima entre diferencias finitas y autograd: 1.4923582369874566e-09


La diferencia es del orden de $10^{-9}$: los dos métodos coinciden en muchas cifras. Las diferencias finitas son una aproximación; autograd es exacto salvo por el redondeo de la aritmética de punto flotante.

PyTorch trae esta verificación en `torch.autograd.gradcheck`, útil cuando escribes una operación propia con su `backward`.

## 10. `detach()` y `no_grad()`: cortar el grafo

A veces se quiere que un valor participe del cálculo **como si fuera una constante**: que se use su valor, pero que no se derive a través de él.

- `x.detach()` devuelve un tensor con los mismos valores, desconectado del grafo. Matemáticamente, lo que venga de ahí tiene derivada cero.
- `with torch.no_grad():` desactiva el registro para todo lo que se calcule dentro del bloque.

Ejemplo: $y = x \cdot x$ tiene derivada $2x$. Si uno de los dos factores se desconecta, se trata como constante $c$ y queda $y = c \cdot x$, con derivada $c$.

In [12]:
x = torch.tensor(3.0, requires_grad=True)

y = x * x
y.backward()
print("d(x·x)/dx          =", x.grad.item())

x.grad.zero_()
y = x.detach() * x              # el primer factor es una constante para autograd
y.backward()
print("d(x.detach()·x)/dx =", x.grad.item())

d(x·x)/dx          = 6.0
d(x.detach()·x)/dx = 3.0


El valor de $y$ es 9 en los dos casos; lo que cambia es la derivada. Esto explica dos líneas que aparecen en todos los notebooks anteriores:

- La actualización `w -= lr * w.grad` va dentro de `no_grad()` porque es un ajuste de los parámetros y no una parte del modelo. Si quedara registrada, el siguiente `backward()` intentaría derivar a través de ella.
- Al predecir se usa `no_grad()` porque no se va a derivar nada, y registrar el grafo cuesta memoria.

## 11. Resumen

- El **gradiente** es el caso de muchas entradas y una salida; el **jacobiano**, de muchas entradas y muchas salidas.
- La **regla de la cadena** dice que los jacobianos de una composición se multiplican, sumando sobre todos los caminos intermedios.
- PyTorch registra las operaciones en un **grafo** y `backward()` lo recorre hacia atrás calculando **productos vector-jacobiano**. Nunca construye un jacobiano completo.
- El modo reverso entrega todas las derivadas de una pérdida escalar en una sola pasada. Por eso `backward()` necesita un escalar.
- Los gradientes **se acumulan** porque la regla de la cadena suma sobre caminos. De ahí `zero_grad()`.
- Para el producto matricial $C = AB$: $\partial L / \partial A = G B^\top$ y $\partial L / \partial B = A^\top G$.
- La retropropagación de una red es la aplicación ordenada de esas reglas locales.
- Las **diferencias finitas** permiten verificar cualquier gradiente.

## Ejercicios

1. Calcula a mano el jacobiano de $f(x_1, x_2) = (x_1 e^{x_2}, \; x_1^2 + x_2^2)$ y compáralo con `jacobian`.
2. En la sección 8, cambia la ReLU por `torch.tanh`. ¿Qué paso de la tabla cambia? Escribe la fórmula nueva (la derivada de $\tanh$ es $1 - \tanh^2$) y comprueba contra autograd.
3. Agrega una segunda capa oculta a la red de la sección 8 y extiende la retropropagación a mano.
4. En la sección 9, repite la verificación con `eps` igual a `1e-2`, `1e-4`, `1e-8` y `1e-12`. ¿Para qué valores empeora el error? ¿Por qué empeora también con `eps` muy pequeño?
5. Repite la sección 9 con diferencias de un solo lado, $(f(\theta + \varepsilon e_i) - f(\theta)) / \varepsilon$. ¿Cuánto mayor es el error?
6. Usa `torch.autograd.gradcheck` para verificar la función `capa` de la sección 4.